# ARC-AGI-2 Neuro-Symbolic Solver (submission-ready)
LLM-proposes (Qwen2.5-VL-7B, 4-bit) + symbolic verifier-checks. The `arc_agi2`
package is shipped as files (a dataset at /kaggle/input/arc-agi-2-pkg) and
imported normally, so the notebook runs fully offline at eval time.

**Score rule:** each test input gets `{attempt_1, attempt_2}`; the task scores
if EITHER matches ground truth exactly. We use attempt_2 as an identity fallback
so the grid is never empty.

**Accelerator:** select L4x4 (96GB) for the 7B model (~15GB in 4-bit).

In [ ]:
import os, json, time, sys, base64
from pathlib import Path

# Self-contained: (re)create the arc_agi2 package under /kaggle/working from the
# embedded sources below, so the notebook needs NO external dataset mount. We
# ALWAYS overwrite (a stale partial package from a prior run would otherwise be
# imported as a broken namespace package).
_PKG_DIR = Path("/kaggle/working/arc_agi2")
import shutil as _shutil
if _PKG_DIR.exists():
    _shutil.rmtree(_PKG_DIR)
_PKG_DIR.mkdir(parents=True, exist_ok=True)
_MODULES = {
    'loader.py': 'IiIiRGF0YSBsb2FkaW5nIGZvciBBUkMtQUdJLTIuCgpBUkMtQUdJLTIgc2hpcHMgcGVyIHRoZSBvZmZpY2lhbCBzcGVjICgyMDI2KSB3aXRoIHRoZXNlIGZpbGVzOgogIGFyYy1hZ2lfdHJhaW5pbmctY2hhbGxlbmdlcy5qc29uICAgKyBhcmMtYWdpX3RyYWluaW5nLXNvbHV0aW9ucy5qc29uCiAgYXJjLWFnaV9ldmFsdWF0aW9uLWNoYWxsZW5nZXMuanNvbiArIGFyYy1hZ2lfZXZhbHVhdGlvbi1zb2x1dGlvbnMuanNvbgogIGFyYy1hZ2lfdGVzdC1jaGFsbGVuZ2VzLmpzb24gICAgICAgKG91dHB1dHMgd2l0aGhlbGQ7IHByaXZhdGUgYXQgZXZhbCkKICBzYW1wbGVfc3VibWlzc2lvbi5qc29uCgpUaGUgbG9hZGVyIGlzIHBhdGgtYWdub3N0aWM6IHBvaW50IGl0IGF0IHRoZSBkaXJlY3RvcnkgaG9sZGluZyB0aGVzZSBmaWxlcwooZW52IEFSQ19EQVRBX0RJUiBvciB0aGUgYGRhdGFfZGlyYCBhcmd1bWVudCkuIEZpbGVuYW1lcyBhcmUgbWF0Y2hlZCBieQpnbG9iIHNvIG1pbm9yIG5hbWluZyBkaWZmZXJlbmNlcyBhcmUgdG9sZXJhdGVkLgoiIiIKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGpzb24KaW1wb3J0IG9zCmZyb20gZGF0YWNsYXNzZXMgaW1wb3J0IGRhdGFjbGFzcywgZmllbGQKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmZyb20gdHlwaW5nIGltcG9ydCBBbnkKCkRBVEFfRElSX0VOViA9ICJBUkNfREFUQV9ESVIiCgoKQGRhdGFjbGFzcwpjbGFzcyBUYXNrOgogICAgdGFza19pZDogc3RyCiAgICB0cmFpbjogbGlzdFtkaWN0XSA9IGZpZWxkKGRlZmF1bHRfZmFjdG9yeT1saXN0KSAgICMgW3siaW5wdXQiLCJvdXRwdXQifV0KICAgIHRlc3Q6IGxpc3RbZGljdF0gPSBmaWVsZChkZWZhdWx0X2ZhY3Rvcnk9bGlzdCkgICAgIyBbeyJpbnB1dCJ9XSAoKyAib3V0cHV0IiBpZiBrbm93bikKCiAgICBAcHJvcGVydHkKICAgIGRlZiBuX3RyYWluKHNlbGYpIC0+IGludDoKICAgICAgICByZXR1cm4gbGVuKHNlbGYudHJhaW4pCgogICAgQHByb3BlcnR5CiAgICBkZWYgbl90ZXN0KHNlbGYpIC0+IGludDoKICAgICAgICByZXR1cm4gbGVuKHNlbGYudGVzdCkKCgpkZWYgX2ZpbmRfZmlsZShkYXRhX2RpcjogUGF0aCwgcGF0dGVybnM6IGxpc3Rbc3RyXSkgLT4gUGF0aCB8IE5vbmU6CiAgICAjIEJlIHRvbGVyYW50IG9mIGh5cGhlbi91bmRlcnNjb3JlIHNlcGFyYXRvcnMgaW4gdGhlIG9mZmljaWFsIGZpbGVuYW1lcywKICAgICMgZS5nLiBhcmMtYWdpX2V2YWx1YXRpb24tY2hhbGxlbmdlcy5qc29uIHZzIGFyYy1hZ2lfZXZhbHVhdGlvbl9jaGFsbGVuZ2VzLmpzb24uCiAgICBpbXBvcnQgcmUKICAgIGFsbF9wYXR0ZXJucyA9IFtdCiAgICBmb3IgcGF0IGluIHBhdHRlcm5zOgogICAgICAgIGFsbF9wYXR0ZXJucy5hcHBlbmQocGF0KQogICAgICAgICMgdmFyaWFudCB3aXRoIHNlcGFyYXRvcnMgZmxpcHBlZCBiZXR3ZWVuIHdvcmRzCiAgICAgICAgdmFyaWFudCA9IHJlLnN1YihyIlstX10iLCBsYW1iZGEgbTogIl8iIGlmIG0uZ3JvdXAoMCkgPT0gIi0iIGVsc2UgIi0iLCBwYXQpCiAgICAgICAgaWYgdmFyaWFudCAhPSBwYXQ6CiAgICAgICAgICAgIGFsbF9wYXR0ZXJucy5hcHBlbmQodmFyaWFudCkKICAgIGZvciBwYXQgaW4gYWxsX3BhdHRlcm5zOgogICAgICAgIGhpdHMgPSBzb3J0ZWQoZGF0YV9kaXIuZ2xvYihwYXQpKQogICAgICAgIGlmIGhpdHM6CiAgICAgICAgICAgIHJldHVybiBoaXRzWzBdCiAgICByZXR1cm4gTm9uZQoKCmRlZiBfcmVzb2x2ZV9kYXRhX2RpcihkYXRhX2Rpcjogc3RyIHwgb3MuUGF0aExpa2UgfCBOb25lID0gTm9uZSkgLT4gUGF0aDoKICAgIGlmIGRhdGFfZGlyIGlzIE5vbmU6CiAgICAgICAgZGF0YV9kaXIgPSBvcy5lbnZpcm9uLmdldChEQVRBX0RJUl9FTlYpCiAgICBpZiBkYXRhX2RpciBpcyBOb25lOgogICAgICAgIHJhaXNlIEZpbGVOb3RGb3VuZEVycm9yKAogICAgICAgICAgICBmIk5vIGRhdGEgZGlyZWN0b3J5IGdpdmVuIGFuZCAke0RBVEFfRElSX0VOVn0gaXMgbm90IHNldC4gIgogICAgICAgICAgICAiUGFzcyBkYXRhX2Rpcj0uLi4gb3IgZXhwb3J0IEFSQ19EQVRBX0RJUj0vcGF0aC90by9hcmMtYWdpLTIvZGF0YSIKICAgICAgICApCiAgICBwID0gUGF0aChkYXRhX2RpcikKICAgIGlmIG5vdCBwLmV4aXN0cygpOgogICAgICAgIHJhaXNlIEZpbGVOb3RGb3VuZEVycm9yKGYiRGF0YSBkaXJlY3RvcnkgZG9lcyBub3QgZXhpc3Q6IHtwfSIpCiAgICByZXR1cm4gcAoKCmRlZiBsb2FkX2NoYWxsZW5nZXMoZGF0YV9kaXI9Tm9uZSwgc3BsaXQ6IHN0ciA9ICJldmFsdWF0aW9uIikgLT4gZGljdFtzdHIsIEFueV06CiAgICAiIiJzcGxpdCBpbiB7dHJhaW5pbmcsIGV2YWx1YXRpb24sIHRlc3R9LiBSZXR1cm5zIHJhdyBjaGFsbGVuZ2UgZGljdC4iIiIKICAgIGQgPSBfcmVzb2x2ZV9kYXRhX2RpcihkYXRhX2RpcikKICAgIG5hbWUgPSAidGVzdCIgaWYgc3BsaXQgPT0gInRlc3QiIGVsc2Ugc3BsaXQKICAgIGYgPSBfZmluZF9maWxlKGQsIFtmIip7bmFtZX0qLWNoYWxsZW5nZXMuanNvbiIsIGYiKntzcGxpdH0qLWNoYWxsZW5nZXMuanNvbiJdKQogICAgaWYgZiBpcyBOb25lOgogICAgICAgIHJhaXNlIEZpbGVOb3RGb3VuZEVycm9yKGYiQ291bGQgbm90IGZpbmQge3NwbGl0fSBjaGFsbGVuZ2VzIEpTT04gaW4ge2R9IikKICAgIHJldHVybiBqc29uLmxvYWRzKGYucmVhZF90ZXh0KCkpCgoKZGVmIGxvYWRfc29sdXRpb25zKGRhdGFfZGlyPU5vbmUsIHNwbGl0OiBzdHIgPSAiZXZhbHVhdGlvbiIpIC0+IGRpY3Rbc3RyLCBBbnldOgogICAgZCA9IF9yZXNvbHZlX2RhdGFfZGlyKGRhdGFfZGlyKQogICAgZiA9IF9maW5kX2ZpbGUoZCwgW2YiKntzcGxpdH0qLXNvbHV0aW9ucy5qc29uIl0pCiAgICBpZiBmIGlzIE5vbmU6CiAgICAgICAgcmFpc2UgRmlsZU5vdEZvdW5kRXJyb3IoZiJDb3VsZCBub3QgZmluZCB7c3BsaXR9IHNvbHV0aW9ucyBKU09OIGluIHtkfSIpCiAgICByZXR1cm4ganNvbi5sb2FkcyhmLnJlYWRfdGV4dCgpKQoKCmRlZiBsb2FkX3Rhc2sodGFza19pZDogc3RyLCBkYXRhX2Rpcj1Ob25lLCBzcGxpdDogc3RyID0gImV2YWx1YXRpb24iKSAtPiBUYXNrOgogICAgY2hhbGxlbmdlcyA9IGxvYWRfY2hhbGxlbmdlcyhkYXRhX2Rpciwgc3BsaXQpCiAgICBpZiB0YXNrX2lkIG5vdCBpbiBjaGFsbGVuZ2VzOgogICAgICAgIHJhaXNlIEtleUVycm9yKGYie3Rhc2tfaWR9IG5vdCBpbiB7c3BsaXR9IGNoYWxsZW5nZXMiKQogICAgcmF3ID0gY2hhbGxlbmdlc1t0YXNrX2lkXQogICAgcmV0dXJuIFRhc2sodGFza19pZD10YXNrX2lkLCB0cmFpbj1yYXcuZ2V0KCJ0cmFpbiIsIFtdKSwgdGVzdD1yYXcuZ2V0KCJ0ZXN0IiwgW10pKQoKCmRlZiBsb2FkX2FsbChkYXRhX2Rpcj1Ob25lLCBzcGxpdDogc3RyID0gImV2YWx1YXRpb24iKSAtPiBkaWN0W3N0ciwgVGFza106CiAgICAiIiJMb2FkIGV2ZXJ5IHRhc2sgZm9yIGEgc3BsaXQuIEZvciB0cmFpbmluZy9ldmFsdWF0aW9uIHRoZSBzb2x1dGlvbnMgYXJlCiAgICBtZXJnZWQgaW50byB0aGUgdGVzdCBwYWlycycgJ291dHB1dCcgZmllbGQgd2hlbiBwcmVzZW50IChzbyB3ZSBjYW4gc2NvcmUKICAgIGxvY2FsbHkpLiBUZXN0IHNwbGl0IChwcml2YXRlKSBoYXMgbm8gb3V0cHV0cyAtPiBsZWZ0IE5vbmUuIiIiCiAgICBjaGFsbGVuZ2VzID0gbG9hZF9jaGFsbGVuZ2VzKGRhdGFfZGlyLCBzcGxpdCkKICAgIG91dDogZGljdFtzdHIsIFRhc2tdID0ge30KICAgIHNvbCA9IE5vbmUKICAgIGlmIHNwbGl0IGluICgidHJhaW5pbmciLCAiZXZhbHVhdGlvbiIpOgogICAgICAgIHRyeToKICAgICAgICAgICAgc29sID0gbG9hZF9zb2x1dGlvbnMoZGF0YV9kaXIsIHNwbGl0KQogICAgICAgIGV4Y2VwdCBGaWxlTm90Rm91bmRFcnJvcjoKICAgICAgICAgICAgc29sID0gTm9uZQogICAgZm9yIHRpZCwgcmF3IGluIGNoYWxsZW5nZXMuaXRlbXMoKToKICAgICAgICB0ZXN0X3BhaXJzID0gW2RpY3QocCkgZm9yIHAgaW4gcmF3LmdldCgidGVzdCIsIFtdKV0KICAgICAgICBpZiBzb2wgaXMgbm90IE5vbmUgYW5kIHRpZCBpbiBzb2w6CiAgICAgICAgICAgIGdvbGQgPSBzb2xbdGlkXQogICAgICAgICAgICBmb3IgaSwgZyBpbiBlbnVtZXJhdGUoZ29sZCk6CiAgICAgICAgICAgICAgICBpZiBpIDwgbGVuKHRlc3RfcGFpcnMpOgogICAgICAgICAgICAgICAgICAgIHRlc3RfcGFpcnNbaV1bIm91dHB1dCJdID0gZwogICAgICAgIG91dFt0aWRdID0gVGFzayh0YXNrX2lkPXRpZCwgdHJhaW49cmF3LmdldCgidHJhaW4iLCBbXSksIHRlc3Q9dGVzdF9wYWlycykKICAgIHJldHVybiBvdXQK',
    'grid_utils.py': 'IiIiR3JpZCByZW5kZXJpbmc6IEFTQ0lJIGZvciBkZWJ1Z2dpbmcsIFBJTCBpbWFnZSBmb3IgUXdlbjIuNS1WTCBpbnB1dC4KCkFSQyBncmlkcyBhcmUgbGlzdC1vZi1saXN0cyBvZiBpbnRzIDAtOS4gV2UgcmVuZGVyIGVhY2ggY29sb3IgYXMgYSBkaXN0aW5jdApmaWxsZWQgY2VsbCBzbyB0aGUgdmlzaW9uIG1vZGVsIGNhbiByZWFkIHRoZW0uIDAgaXMgdHJlYXRlZCBhcyBiYWNrZ3JvdW5kLgoiIiIKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBQSUwgaW1wb3J0IEltYWdlLCBJbWFnZURyYXcKCiMgRGlzdGluY3QsIGhpZ2gtY29udHJhc3QgY29sb3JzIGZvciAwLi45IChSR0IpLiBJbmRleCAwID0gYmxhY2sgYmFja2dyb3VuZC4KQVJDX0NPTE9SUyA9IFsKICAgICgwLCAwLCAwKSwgICAgICAgICMgMCBibGFjawogICAgKDI1NSwgMCwgMCksICAgICAgIyAxIHJlZAogICAgKDAsIDIwMCwgMCksICAgICAgIyAyIGdyZWVuCiAgICAoMCwgMTAwLCAyNTUpLCAgICAjIDMgYmx1ZQogICAgKDI1NSwgMjU1LCAwKSwgICAgIyA0IHllbGxvdwogICAgKDI1NSwgMCwgMjU1KSwgICAgIyA1IG1hZ2VudGEKICAgICgwLCAyNTUsIDI1NSksICAgICMgNiBjeWFuCiAgICAoMTgwLCAxODAsIDE4MCksICAjIDcgZ3JleQogICAgKDE1MCwgNzUsIDApLCAgICAgIyA4IGJyb3duCiAgICAoMjU1LCAyNTUsIDI1NSksICAjIDkgd2hpdGUKXQoKQVNDSUlfQ0hBUlMgPSAiIC4xMjM0NTY3ODkiICAjIGluZGV4IGFsaWducyB3aXRoIGNvbG9yIHZhbHVlIDAuLjkKCgpkZWYgZ3JpZF90b19hc2NpaShncmlkOiBsaXN0W2xpc3RbaW50XV0sIHNlcDogc3RyID0gIiAiKSAtPiBzdHI6CiAgICByb3dzID0gW10KICAgIGZvciByIGluIGdyaWQ6CiAgICAgICAgcm93cy5hcHBlbmQoc2VwLmpvaW4oQVNDSUlfQ0hBUlNbdl0gaWYgMCA8PSB2IDwgMTAgZWxzZSAiPyIgZm9yIHYgaW4gcikpCiAgICByZXR1cm4gIlxuIi5qb2luKHJvd3MpCgoKZGVmIGdyaWRfdG9faW1hZ2UoZ3JpZDogbGlzdFtsaXN0W2ludF1dLCBjZWxsOiBpbnQgPSAzMiwgYm9yZGVyOiBpbnQgPSAyKSAtPiBJbWFnZS5JbWFnZToKICAgICIiIlJlbmRlciBhIGdyaWQgYXMgYSBjbGVhbiBQSUwgaW1hZ2Ugd2l0aCBjb2xvcmVkIGNlbGxzLgoKICAgIGNlbGwgIC0gcGl4ZWwgc2l6ZSBvZiBlYWNoIGdyaWQgY2VsbAogICAgYm9yZGVyLSBibGFjayBnYXAgYmV0d2VlbiBjZWxscyBzbyB0aGUgbW9kZWwgY2FuIHNlZSBncmlkIGxpbmVzCiAgICAiIiIKICAgIGggPSBsZW4oZ3JpZCkKICAgIHcgPSBsZW4oZ3JpZFswXSkgaWYgaCBlbHNlIDAKICAgIGlmIHcgPT0gMDoKICAgICAgICByZXR1cm4gSW1hZ2UubmV3KCJSR0IiLCAoY2VsbCwgY2VsbCksICgwLCAwLCAwKSkKICAgIGltZyA9IEltYWdlLm5ldygiUkdCIiwgKHcgKiBjZWxsLCBoICogY2VsbCksICgyMCwgMjAsIDIwKSkKICAgIGRyYXcgPSBJbWFnZURyYXcuRHJhdyhpbWcpCiAgICBmb3IgeSwgcm93IGluIGVudW1lcmF0ZShncmlkKToKICAgICAgICBmb3IgeCwgdiBpbiBlbnVtZXJhdGUocm93KToKICAgICAgICAgICAgY29sb3IgPSBBUkNfQ09MT1JTW3ZdIGlmIDAgPD0gdiA8IDEwIGVsc2UgKDgwLCA4MCwgODApCiAgICAgICAgICAgIGRyYXcucmVjdGFuZ2xlKAogICAgICAgICAgICAgICAgW3ggKiBjZWxsICsgYm9yZGVyLCB5ICogY2VsbCArIGJvcmRlciwKICAgICAgICAgICAgICAgICAoeCArIDEpICogY2VsbCAtIGJvcmRlciwgKHkgKyAxKSAqIGNlbGwgLSBib3JkZXJdLAogICAgICAgICAgICAgICAgZmlsbD1jb2xvciwKICAgICAgICAgICAgKQogICAgcmV0dXJuIGltZwoKCmRlZiByZW5kZXJfdGFza190aHVtYm5haWxzKHRhc2ssIGNlbGw6IGludCA9IDI0KSAtPiBsaXN0W0ltYWdlLkltYWdlXToKICAgICIiIlJlbmRlciBldmVyeSB0cmFpbi90ZXN0IHBhaXIgKGlucHV0K291dHB1dCkgYXMgaW1hZ2VzIGZvciB0aGUgbW9kZWwgcHJvbXB0LiIiIgogICAgaW1nczogbGlzdFtJbWFnZS5JbWFnZV0gPSBbXQogICAgZm9yIGksIHBhaXIgaW4gZW51bWVyYXRlKHRhc2sudHJhaW4pOgogICAgICAgIGltZ3MuYXBwZW5kKGdyaWRfdG9faW1hZ2UocGFpclsiaW5wdXQiXSwgY2VsbCkpCiAgICAgICAgaWYgIm91dHB1dCIgaW4gcGFpcjoKICAgICAgICAgICAgaW1ncy5hcHBlbmQoZ3JpZF90b19pbWFnZShwYWlyWyJvdXRwdXQiXSwgY2VsbCkpCiAgICBmb3IgcGFpciBpbiB0YXNrLnRlc3Q6CiAgICAgICAgaW1ncy5hcHBlbmQoZ3JpZF90b19pbWFnZShwYWlyWyJpbnB1dCJdLCBjZWxsKSkKICAgIHJldHVybiBpbWdzCg==',
    'dsl.py': '...',
    'verifier.py': 'IiIiVmVyaWZpZXI6IHJ1biBhIGNhbmRpZGF0ZSBwcm9ncmFtIGFnYWluc3QgdHJhaW4gcGFpcnMsIGNoZWNrIGV4YWN0IG1hdGNoLgoKQSAicHJvZ3JhbSIgaXMgYSBQeXRob24gY2FsbGFibGUgKGdyaWQgLT4gZ3JpZCkgU09VUkNFRCBmcm9tIGEgcmVzdHJpY3RlZApuYW1lc3BhY2UuIFRoaXMgbW9kdWxlIHByb3ZpZGVzOgogIC0gcnVuX3Byb2dyYW0oc3JjLCBncmlkKTogZXZhbCBhIERTTCBzdHJpbmcgc2FmZWx5IGFuZCBhcHBseSBpdCB0byBhIGdyaWQKICAtIHZlcmlmeV9wcm9ncmFtKHNyYywgdGFzayk6IGFwcGx5IHRvIGFsbCB0cmFpbiBpbnB1dHM7IFRydWUgaWZmIGV2ZXJ5CiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgb3V0cHV0IG1hdGNoZXMgdGhlIHRyYWluIG91dHB1dCBleGFjdGx5LgoKVGhlIERTTCBzdHJpbmcgaXMgd2hhdCB0aGUgTExNL1ZMTSBwcm9wb3NlczsgdGhlIHZlcmlmaWVyIGlzIHRoZSBzeW1ib2xpYwpnYXRlIHRoYXQgbXVzdCBwYXNzIGJlZm9yZSB3ZSB0cnVzdCB0aGUgcHJvZ3JhbSBvbiB0aGUgdGVzdCBpbnB1dC4KIiIiCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBudW1weSBhcyBucAoKZnJvbSAuZHNsIGltcG9ydCBTQUZFX0dMT0JBTFMKCgpkZWYgcnVuX3Byb2dyYW0oc3JjOiBzdHIsIGdyaWQpIC0+IGxpc3RbbGlzdFtpbnRdXToKICAgICIiIkFwcGx5IGEgRFNMIHNvdXJjZSBzdHJpbmcgdG8gYSBzaW5nbGUgZ3JpZC4gUmV0dXJucyBsaXN0W2xpc3RbaW50XV0uIiIiCiAgICBnID0gbnAuYXJyYXkoZ3JpZCwgZHR5cGU9aW50KQogICAgbmFtZXNwYWNlOiBkaWN0ID0ge30KICAgIGV4ZWMoY29tcGlsZShzcmMsICI8cHJvZ3JhbT4iLCAiZXhlYyIpLCBkaWN0KFNBRkVfR0xPQkFMUyksIG5hbWVzcGFjZSkKICAgIGlmICJzb2x2ZSIgbm90IGluIG5hbWVzcGFjZToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJwcm9ncmFtIG11c3QgZGVmaW5lIHNvbHZlKGc6IG5kYXJyYXkpIC0+IG5kYXJyYXkiKQogICAgb3V0ID0gbmFtZXNwYWNlWyJzb2x2ZSJdKGcpCiAgICBvdXQgPSBucC5hcnJheShvdXQsIGR0eXBlPWludCkKICAgIHJldHVybiBvdXQudG9saXN0KCkKCgpkZWYgdmVyaWZ5X3Byb2dyYW0oc3JjOiBzdHIsIHRhc2ssIHZlcmJvc2U6IGJvb2wgPSBGYWxzZSkgLT4gYm9vbDoKICAgICIiIlJldHVybiBUcnVlIGlmZiB0aGUgcHJvZ3JhbSByZXByb2R1Y2VzIGV2ZXJ5IHRyYWluIG91dHB1dCBleGFjdGx5LiIiIgogICAgdHJ5OgogICAgICAgIGZvciBpLCBwYWlyIGluIGVudW1lcmF0ZSh0YXNrLnRyYWluKToKICAgICAgICAgICAgcHJlZCA9IHJ1bl9wcm9ncmFtKHNyYywgcGFpclsiaW5wdXQiXSkKICAgICAgICAgICAgZ29sZCA9IG5wLmFycmF5KHBhaXJbIm91dHB1dCJdLCBkdHlwZT1pbnQpCiAgICAgICAgICAgIGlmIG5vdCBucC5hcnJheV9lcXVhbChucC5hcnJheShwcmVkKSwgZ29sZCk6CiAgICAgICAgICAgICAgICBpZiB2ZXJib3NlOgogICAgICAgICAgICAgICAgICAgIHByaW50KGYiICB0cmFpbiBwYWlyIHtpfTogTUlTTUFUQ0giKQogICAgICAgICAgICAgICAgcmV0dXJuIEZhbHNlCiAgICAgICAgcmV0dXJuIFRydWUKICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZTogICMgbWFsZm9ybWVkIHByb2dyYW0gLT4gZmFpbHMgdmVyaWZpY2F0aW9uCiAgICAgICAgaWYgdmVyYm9zZToKICAgICAgICAgICAgcHJpbnQoZiIgIHByb2dyYW0gZXJyb3I6IHt0eXBlKGUpLl9fbmFtZV9ffToge2V9IikKICAgICAgICByZXR1cm4gRmFsc2UK',
    'submission.py': 'IiIiU3VibWlzc2lvbiB3cml0aW5nIGZvciBBUkMtQUdJLTIuCgpGb3JtYXQgKG9mZmljaWFsIHNwZWMpOgogIHN1Ym1pc3Npb24uanNvbiA9IHsKICAgICAiPHRhc2tfaWQ+IjogWyB7ImF0dGVtcHRfMSI6IGdyaWQsICJhdHRlbXB0XzIiOiBncmlkfSwgLi4uIF0sICAjIG9uZSBwZXIgdGVzdCBpbnB1dAogIH0KICBFYWNoIGdyaWQgaXMgbGlzdFtsaXN0W2ludF1dIHdpdGggaW50cyAwLTkuCiAgQk9USCBhdHRlbXB0XzEgYW5kIGF0dGVtcHRfMiByZXF1aXJlZCBmb3IgZXZlcnkgdGVzdCBpbnB1dCwgZXZlbiBpZiBkdW1teS4KIiIiCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBqc29uCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKCmRlZiBlbXB0eV9zdWJtaXNzaW9uKHRhc2tfaWRzOiBsaXN0W3N0cl0sIG5fdGVzdF9wZXJfaWQ6IGRpY3Rbc3RyLCBpbnRdKSAtPiBkaWN0OgogICAgIiIiQnVpbGQgYSBzdWJtaXNzaW9uIHNrZWxldG9uIGZpbGxlZCB3aXRoIGVtcHR5IDF4MSBncmlkcyAodmFsaWQgZHVtbXkpLiIiIgogICAgc3ViID0ge30KICAgIGZvciB0aWQgaW4gdGFza19pZHM6CiAgICAgICAgc3ViW3RpZF0gPSBbCiAgICAgICAgICAgIHsiYXR0ZW1wdF8xIjogW1swXV0sICJhdHRlbXB0XzIiOiBbWzBdXX0KICAgICAgICAgICAgZm9yIF8gaW4gcmFuZ2Uobl90ZXN0X3Blcl9pZC5nZXQodGlkLCAxKSkKICAgICAgICBdCiAgICByZXR1cm4gc3ViCgoKZGVmIHdyaXRlX3N1Ym1pc3Npb24ocGF0aCwgdGFza19wcmVkaWN0aW9uczogZGljdFtzdHIsIGxpc3RbbGlzdFtsaXN0W2ludF1dXV0pOgogICAgIiIidGFza19wcmVkaWN0aW9uczogdGlkIC0+IGxpc3Qgb2YgcHJlZGljdGlvbnMsIG9uZSBwZXIgdGVzdCBpbnB1dC4KICAgIEVhY2ggcHJlZGljdGlvbiBpcyBhIDItdHVwbGUvbGlzdCBbYXR0ZW1wdF8xX2dyaWQsIGF0dGVtcHRfMl9ncmlkXS4KICAgIFdyaXRlcyBzdWJtaXNzaW9uLmpzb24gaW4gdGhlIG9mZmljaWFsIGZvcm1hdC4gUmV0dXJucyB0aGUgZGljdC4iIiIKICAgIHN1YiA9IHt9CiAgICBmb3IgdGlkLCBwcmVkcyBpbiB0YXNrX3ByZWRpY3Rpb25zLml0ZW1zKCk6CiAgICAgICAgc3ViW3RpZF0gPSBbXQogICAgICAgIGZvciBwIGluIHByZWRzOgogICAgICAgICAgICBhMSwgYTIgPSBwWzBdLCBwWzFdCiAgICAgICAgICAgIHN1Ylt0aWRdLmFwcGVuZCh7ImF0dGVtcHRfMSI6IGExLCAiYXR0ZW1wdF8yIjogYTJ9KQogICAgUGF0aChwYXRoKS53cml0ZV90ZXh0KGpzb24uZHVtcHMoc3ViKSkKICAgIHJldHVybiBzdWIKCgpkZWYgdmFsaWRhdGVfc3VibWlzc2lvbihzdWI6IGRpY3QsIHRhc2tfaWRzOiBsaXN0W3N0cl0sIG5fdGVzdF9wZXJfaWQ6IGRpY3Rbc3RyLCBpbnRdKSAtPiBsaXN0W3N0cl06CiAgICAiIiJSZXR1cm4gYSBsaXN0IG9mIHByb2JsZW1zIChlbXB0eSBpZiB2YWxpZCkuIiIiCiAgICBlcnJzID0gW10KICAgIGZvciB0aWQgaW4gdGFza19pZHM6CiAgICAgICAgaWYgdGlkIG5vdCBpbiBzdWI6CiAgICAgICAgICAgIGVycnMuYXBwZW5kKGYibWlzc2luZyB0YXNrIHt0aWR9IikKICAgICAgICAgICAgY29udGludWUKICAgICAgICBwcmVkcyA9IHN1Ylt0aWRdCiAgICAgICAgaWYgbGVuKHByZWRzKSAhPSBuX3Rlc3RfcGVyX2lkLmdldCh0aWQsIDEpOgogICAgICAgICAgICBlcnJzLmFwcGVuZChmInt0aWR9OiBleHBlY3RlZCB7bl90ZXN0X3Blcl9pZC5nZXQodGlkLDEpfSBwcmVkaWN0aW9ucywgZ290IHtsZW4ocHJlZHMpfSIpCiAgICAgICAgZm9yIGksIHAgaW4gZW51bWVyYXRlKHByZWRzKToKICAgICAgICAgICAgaWYgImF0dGVtcHRfMSIgbm90IGluIHAgb3IgImF0dGVtcHRfMiIgbm90IGluIHA6CiAgICAgICAgICAgICAgICBlcnJzLmFwcGVuZChmInt0aWR9IHByZWQge2l9OiBtaXNzaW5nIGF0dGVtcHQga2V5cyIpCiAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICBmb3IgayBpbiAoImF0dGVtcHRfMSIsICJhdHRlbXB0XzIiKToKICAgICAgICAgICAgICAgIGcgPSBwW2tdCiAgICAgICAgICAgICAgICBpZiBub3QgKGlzaW5zdGFuY2UoZywgbGlzdCkgYW5kIGcgYW5kIGlzaW5zdGFuY2UoZ1swXSwgbGlzdCkpOgogICAgICAgICAgICAgICAgICAgIGVycnMuYXBwZW5kKGYie3RpZH0gcHJlZCB7aX0ge2t9OiBub3QgYSAyRCBncmlkIikKICAgIHJldHVybiBlcnJzCg==',
    'models.py': 'IiIiUXdlbjIuNS1WTCBpbnRlcmZhY2UgZm9yIEFSQy1BR0ktMiAoYnVuZGxlZCwgb2ZmbGluZSBvbiBLYWdnbGUpLgoKTEFaWSBpbXBvcnRzOiB0b3JjaC90cmFuc2Zvcm1lcnMgb25seSBpbXBvcnRlZCBpbnNpZGUgZnVuY3Rpb25zLCBzbyB0aGUgcmVzdApvZiB0aGUgbGlicmFyeSBpbXBvcnRzIGNsZWFubHkgb24gQ1BVLW9ubHkgbWFjaGluZXMgKFdTTCwgbm8gR1BVKS4KCkRlc2lnbiAobmV1cm8tc3ltYm9saWMpOgogIDEuIFJlbmRlciB0cmFpbiBwYWlycyArIHRlc3QgaW5wdXQgYXMgaW1hZ2VzLgogIDIuIEFzayB0aGUgVkxNIHRvIFdSSVRFIGEgUHl0aG9uIGBzb2x2ZShnKWAgZnVuY3Rpb24gKGcgPSBudW1weSBpbnQgYXJyYXkpLgogIDMuIHByb3Bvc2UoKSByZXR1cm5zIHNldmVyYWwgY2FuZGlkYXRlIHNvdXJjZSBzdHJpbmdzIChzYW1wbGVkLCB0ZW1wPjApLgogIDQuIFRoZSBOT1RFQk9PSyB2ZXJpZmllcyBlYWNoIGNhbmRpZGF0ZSB3aXRoIHZlcmlmaWVyLnZlcmlmeV9wcm9ncmFtIGFnYWluc3QKICAgICB0aGUgdHJhaW4gcGFpcnM7IHRoZSBmaXJzdCB0aGF0IHJlcHJvZHVjZXMgYWxsIHRyYWluIG91dHB1dHMgZXhhY3RseSBpcwogICAgIHVzZWQgdG8gZ2VuZXJhdGUgdGhlIHRlc3QgcHJlZGljdGlvbi4gVW52ZXJpZmllZCBwcm9wb3NhbHMgYXJlIGRpc2NhcmRlZC4KClRoaXMga2VlcHMgdGhlIG1vZGVsIHByb3Bvc2VyIGFuZCB0aGUgc3ltYm9saWMgdmVyaWZpZXIgc3RyaWN0bHkgc2VwYXJhdGVkOgp0aGUgbW9kZWwgY2FuIGhhbGx1Y2luYXRlIGZyZWVseTsgb25seSB2ZXJpZmllZCBwcm9ncmFtcyByZWFjaCB0aGUgc3VibWlzc2lvbi4KIiIiCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCByZQppbXBvcnQgdGV4dHdyYXAKZnJvbSB0eXBpbmcgaW1wb3J0IE9wdGlvbmFsCgpTQUZFX0NPTE9SU19OT1RFID0gKAogICAgIkdyaWQgY29sb3JzIGFyZSBpbnRzIDAtOTsgMCBpcyB0aGUgYmxhY2sgYmFja2dyb3VuZC4gIgogICAgIkdyaWRzIGFyZSByZW5kZXJlZCBhcyBjb2xvcmVkIGNlbGxzIGluIHRoZSBpbWFnZXMuIgopCgpTWVNURU1fUFJPTVBUID0gdGV4dHdyYXAuZGVkZW50KAogICAgIiIiCiAgICBZb3UgYXJlIGFuIEFSQy1BR0kgZ3JpZC1yZWFzb25pbmcgc29sdmVyLiBZb3UgcmVjZWl2ZSBleGFtcGxlIGlucHV0L291dHB1dAogICAgZ3JpZHMgYXMgaW1hZ2VzIGFuZCBvbmUgdGVzdCBpbnB1dCBncmlkLiBJbmZlciB0aGUgdHJhbnNmb3JtYXRpb24gcnVsZQogICAgYW5kIFdSSVRFIGEgUHl0aG9uIGZ1bmN0aW9uIGBzb2x2ZShnKWAgd2hlcmUgYGdgIGlzIGEgbnVtcHkgaW50IGFycmF5CiAgICAocm93cyB4IGNvbHMpLiBBdmFpbGFibGUgaGVscGVycyAoYWxyZWFkeSBpbiBzY29wZSk6CgogICAgICBPUklFTlRBVElPTjoKICAgICAgICByb3RhdGVfY3csIHJvdGF0ZV9jY3csIGZsaXBfaCwgZmxpcF92LCB0cmFuc3Bvc2UKICAgICAgQ09MT1I6CiAgICAgICAgaW52ZXJ0X2NvbG9ycywgY29sb3JfcmVwbGFjZShnLCBzcmMsIGRzdCksIGtlZXBfY29sb3IoZywgYykKICAgICAgR0VPTUVUUlk6CiAgICAgICAgY3JvcF9ub256ZXJvLCBib3VuZGluZ19ib3hfY3JvcCwgc2hpZnRfdG9fb3JpZ2luLCBncm93X25vbnplcm8KICAgICAgICBzY2FsZV91cChnLCBrKSwgc2NhbGVfZG93bihnLCBrKQogICAgICBUSUxJTkcgKHVzZSB3aGVuIG91dHB1dCBpcyBhIHNlbGYtc2ltaWxhciBwYXR0ZXJuIG9mIHRoZSBpbnB1dCk6CiAgICAgICAga3Jvbl90aWxlKGcsIGspICAgICAgICAgLSByZXBlYXQgdGhlIGlucHV0IGsgeCBrCiAgICAgICAgbWFza2VkX2tyb25fdGlsZShnLCBrKSAgLSB1c2UgaW5wdXQgYXMgYSBiaW5hcnkgbWFzazsgcGxhY2UgYSBjb3B5CiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBvZiBnIGF0IGVhY2ggbm9uLXplcm8gY2VsbCBvZiB0aGUgbWFzawogICAgICAgIGJyaWNrd2FsbF90aWxlKGcsIGspICAgIC0gayB4IGsgdGlsaW5nIHdpdGggaG9yaXpvbnRhbC1mbGlwIG9uCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBldmVyeSBvZGQgcm93IG9mIHRpbGVzCiAgICAgIFJFR0lPTjoKICAgICAgICBmaW5kX29iamVjdHMoZywgYmcpICAgICAgICAgICAgIC0gbGlzdCBvZiBjb25uZWN0ZWQtY29tcG9uZW50IG1hc2tzCiAgICAgICAgZmlsbF9lbmNsb3NlZChnLCBmcmFtZSwgZmlsbCkgICAtIGZpbGwgemVyby1yZWdpb25zIGVuY2xvc2VkIGJ5CiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGZyYW1lLWNvbG9yICg0LWNvbm5lY3RlZCkKICAgICAgICBmbG9vZF9maWxsXzQoZywgKHksIHgpLCBjb2xvcikgICAtIDQtY29ubmVjdGVkIHBhaW50IGZyb20gc2VlZAoKICAgIFlvdSBtYXkgYWxzbyB3cml0ZSBwbGFpbiBudW1weSAvIFB5dGhvbiBjb2RlLiBgc29sdmUoZylgIG11c3QgcmV0dXJuIGEKICAgIG51bXB5IGludCBhcnJheSAodGhlIG91dHB1dCBncmlkKS4gT3V0cHV0IE9OTFkgcHl0aG9uIGNvZGUgc3RhcnRpbmcKICAgIHdpdGggYGRlZiBzb2x2ZShnKTpgIGFuZCBub3RoaW5nIGVsc2UuIE5vIG1hcmtkb3duLCBubyBjb21tZW50YXJ5LgogICAgIiIiCikuc3RyaXAoKQoKCmRlZiBfZXh0cmFjdF9jb2RlKHRleHQ6IHN0cikgLT4gc3RyOgogICAgIiIiUHVsbCBhIGBkZWYgc29sdmUoZyk6YCBibG9jayBvdXQgb2YgbW9kZWwgb3V0cHV0LCB0b2xlcmF0aW5nIG1hcmtkb3duLiIiIgogICAgIyBTdHJpcCBhIGxlYWRpbmcgc3lzdGVtL2Fzc2lzdGFudCB3cmFwcGVyIGlmIHByZXNlbnQuCiAgICBtID0gcmUuc2VhcmNoKHIiZGVmIHNvbHZlXChnXCk6IiwgdGV4dCkKICAgIGlmIG5vdCBtOgogICAgICAgIHJldHVybiAiIgogICAgc3RhcnQgPSBtLnN0YXJ0KCkKICAgIGJvZHkgPSB0ZXh0W3N0YXJ0Ol0KICAgICMgSWYgZmVuY2VkLCBjdXQgYXQgdGhlIGNsb3NpbmcgZmVuY2UuCiAgICBmZW5jZSA9IHJlLnNlYXJjaChyImBgYCIsIGJvZHlbc3RhcnQgLSAwOl0pICAjIHNlYXJjaCB3aG9sZSBib2R5IGZvciBhIGZlbmNlIGFmdGVyIHN0YXJ0CiAgICAjIFNpbXBsZXI6IGZpbmQgZmlyc3QgYGBgIGFmdGVyIHN0YXJ0CiAgICBmZW5jZV9tYXRjaCA9IHJlLnNlYXJjaChyImBgYCIsIGJvZHkpCiAgICBpZiBmZW5jZV9tYXRjaCBhbmQgZmVuY2VfbWF0Y2guc3RhcnQoKSA+IDA6CiAgICAgICAgIyBvbmx5IHRyZWF0IGFzIGZlbmNlIGlmIGl0IGFwcGVhcnMgYWZ0ZXIgdGhlIGRlZgogICAgICAgIGNhbmQgPSBib2R5W2ZlbmNlX21hdGNoLnN0YXJ0KCk6XQogICAgICAgICMgZW5zdXJlIHRoZSBmZW5jZSBpcyBhZnRlciB0aGUgZGVmIHN0YXJ0CiAgICAgICAgaWYgZmVuY2VfbWF0Y2guc3RhcnQoKSA+IDA6CiAgICAgICAgICAgIGJvZHkgPSBib2R5WzpmZW5jZV9tYXRjaC5zdGFydCgpXQogICAgcmV0dXJuIGJvZHkuc3RyaXAoKQoKCmNsYXNzIFF3ZW5WTDoKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBtb2RlbF9wYXRoOiBzdHIsIGRldmljZTogc3RyID0gImF1dG8iLAogICAgICAgICAgICAgICAgIGxvYWRfaW5fNGJpdDogYm9vbCA9IFRydWUsIGR0eXBlOiBzdHIgPSAiYmZsb2F0MTYiKToKICAgICAgICBzZWxmLm1vZGVsX3BhdGggPSBtb2RlbF9wYXRoCiAgICAgICAgc2VsZi5kZXZpY2UgPSBkZXZpY2UKICAgICAgICBzZWxmLmxvYWRfaW5fNGJpdCA9IGxvYWRfaW5fNGJpdAogICAgICAgIHNlbGYuZHR5cGUgPSBkdHlwZQogICAgICAgIHNlbGYuX21vZGVsID0gTm9uZQogICAgICAgIHNlbGYuX3Byb2Nlc3NvciA9IE5vbmUKCiAgICBkZWYgX2xvYWQoc2VsZik6CiAgICAgICAgaWYgc2VsZi5fbW9kZWwgaXMgbm90IE5vbmU6CiAgICAgICAgICAgIHJldHVybgogICAgICAgIGltcG9ydCB0b3JjaAogICAgICAgIGZyb20gdHJhbnNmb3JtZXJzIGltcG9ydCAoUXdlbjJfNV9WTEZvckNvbmRpdGlvbmFsR2VuZXJhdGlvbiwgQXV0b1Byb2Nlc3NvciwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIEJpdHNBbmRCeXRlc0NvbmZpZykKICAgICAgICBkdCA9IHsiYmZsb2F0MTYiOiB0b3JjaC5iZmxvYXQxNiwgImZsb2F0MTYiOiB0b3JjaC5mbG9hdDE2fS5nZXQoc2VsZi5kdHlwZSwgdG9yY2guYmZsb2F0MTYpCiAgICAgICAga3dhcmdzID0geyJkZXZpY2VfbWFwIjogc2VsZi5kZXZpY2UsICJ0b3JjaF9kdHlwZSI6IGR0fQogICAgICAgIGlmIHNlbGYubG9hZF9pbl80Yml0OgogICAgICAgICAgICBrd2FyZ3NbInF1YW50aXphdGlvbl9jb25maWciXSA9IEJpdHNBbmRCeXRlc0NvbmZpZyhsb2FkX2luXzRiaXQ9VHJ1ZSkKICAgICAgICBzZWxmLl9tb2RlbCA9IFF3ZW4yXzVfVkxGb3JDb25kaXRpb25hbEdlbmVyYXRpb24uZnJvbV9wcmV0cmFpbmVkKAogICAgICAgICAgICBzZWxmLm1vZGVsX3BhdGgsICoqa3dhcmdzCiAgICAgICAgKS5ldmFsKCkKICAgICAgICBzZWxmLl9wcm9jZXNzb3IgPSBBdXRvUHJvY2Vzc29yLmZyb21fcHJldHJhaW5lZChzZWxmLm1vZGVsX3BhdGgpCgogICAgZGVmIF9idWlsZF9tZXNzYWdlcyhzZWxmLCB0YXNrLCBoaW50OiBPcHRpb25hbFtzdHJdID0gTm9uZSk6CiAgICAgICAgZnJvbSAuZ3JpZF91dGlscyBpbXBvcnQgcmVuZGVyX3Rhc2tfdGh1bWJuYWlscwogICAgICAgIGZyb20gUElMIGltcG9ydCBJbWFnZQoKICAgICAgICBpbWdzID0gcmVuZGVyX3Rhc2tfdGh1bWJuYWlscyh0YXNrLCBjZWxsPTMyKQogICAgICAgIGNvbnRlbnQgPSBbXQogICAgICAgIGlkeCA9IDAKICAgICAgICBmb3IgaSwgcGFpciBpbiBlbnVtZXJhdGUodGFzay50cmFpbik6CiAgICAgICAgICAgIGNvbnRlbnQuYXBwZW5kKHsidHlwZSI6ICJpbWFnZSIsICJpbWFnZSI6IGltZ3NbaWR4XX0pCiAgICAgICAgICAgIGNvbnRlbnQuYXBwZW5kKHsidHlwZSI6ICJ0ZXh0IiwgInRleHQiOiBmInRyYWluIHtpfSBJTlBVVCJ9KQogICAgICAgICAgICBpZHggKz0gMQogICAgICAgICAgICBpZiAib3V0cHV0IiBpbiBwYWlyOgogICAgICAgICAgICAgICAgY29udGVudC5hcHBlbmQoeyJ0eXBlIjogImltYWdlIiwgImltYWdlIjogaW1nc1tpZHhdfSkKICAgICAgICAgICAgICAgIGNvbnRlbnQuYXBwZW5kKHsidHlwZSI6ICJ0ZXh0IiwgInRleHQiOiBmInRyYWluIHtpfSBPVVRQVVQifSkKICAgICAgICAgICAgICAgIGlkeCArPSAxCiAgICAgICAgZm9yIGogaW4gcmFuZ2UobGVuKHRhc2sudGVzdCkpOgogICAgICAgICAgICBjb250ZW50LmFwcGVuZCh7InR5cGUiOiAiaW1hZ2UiLCAiaW1hZ2UiOiBpbWdzW2lkeF19KQogICAgICAgICAgICBjb250ZW50LmFwcGVuZCh7InR5cGUiOiAidGV4dCIsICJ0ZXh0IjogZiJ0ZXN0IHtqfSBJTlBVVCAtPiB3cml0ZSBzb2x2ZShnKSJ9KQogICAgICAgICAgICBpZHggKz0gMQogICAgICAgIGNvbnRlbnQuYXBwZW5kKHsidHlwZSI6ICJ0ZXh0IiwgInRleHQiOiBTQUZFX0NPTE9SU19OT1RFfSkKICAgICAgICBpZiBoaW50OgogICAgICAgICAgICBjb250ZW50LmFwcGVuZCh7InR5cGUiOiAidGV4dCIsICJ0ZXh0IjoKICAgICAgICAgICAgICAgICJZb3VyIHByZXZpb3VzIGF0dGVtcHQgZmFpbGVkLiBDb3JyZWN0aW9uIGhpbnQ6ICIgKyBoaW50fSkKICAgICAgICByZXR1cm4gWwogICAgICAgICAgICB7InJvbGUiOiAic3lzdGVtIiwgImNvbnRlbnQiOiBTWVNURU1fUFJPTVBUfSwKICAgICAgICAgICAgeyJyb2xlIjogInVzZXIiLCAiY29udGVudCI6IGNvbnRlbnR9LAogICAgICAgIF0KCiAgICBkZWYgcHJvcG9zZShzZWxmLCB0YXNrLCBuX2NhbmRpZGF0ZXM6IGludCA9IDQsIG1heF9uZXdfdG9rZW5zOiBpbnQgPSAxMDI0LAogICAgICAgICAgICAgICAgdGVtcGVyYXR1cmU6IGZsb2F0ID0gMC43LCBoaW50OiBPcHRpb25hbFtzdHJdID0gTm9uZSkgLT4gbGlzdFtzdHJdOgogICAgICAgICIiIlJldHVybiBhIGxpc3Qgb2YgY2FuZGlkYXRlIGBzb2x2ZShnKWAgc291cmNlIHN0cmluZ3MgKHVudmVyaWZpZWQpLiIiIgogICAgICAgIHNlbGYuX2xvYWQoKQogICAgICAgIG1lc3NhZ2VzID0gc2VsZi5fYnVpbGRfbWVzc2FnZXModGFzaywgaGludD1oaW50KQogICAgICAgIHByb21wdCA9IHNlbGYuX3Byb2Nlc3Nvci5hcHBseV9jaGF0X3RlbXBsYXRlKAogICAgICAgICAgICBtZXNzYWdlcywgdG9rZW5pemU9RmFsc2UsIGFkZF9nZW5lcmF0aW9uX3Byb21wdD1UcnVlCiAgICAgICAgKQogICAgICAgIGltYWdlcyA9IFtjWyJpbWFnZSJdIGZvciBjIGluIG1lc3NhZ2VzWzFdWyJjb250ZW50Il0gaWYgY1sidHlwZSJdID09ICJpbWFnZSJdCiAgICAgICAgaW5wdXRzID0gc2VsZi5fcHJvY2Vzc29yKAogICAgICAgICAgICB0ZXh0PXByb21wdCwgaW1hZ2VzPWltYWdlcywgcmV0dXJuX3RlbnNvcnM9InB0IgogICAgICAgICkudG8oc2VsZi5fbW9kZWwuZGV2aWNlKQogICAgICAgIG91dCA9IHNlbGYuX21vZGVsLmdlbmVyYXRlKAogICAgICAgICAgICAqKmlucHV0cywgbWF4X25ld190b2tlbnM9bWF4X25ld190b2tlbnMsCiAgICAgICAgICAgIGRvX3NhbXBsZT10ZW1wZXJhdHVyZSA+IDAsIHRlbXBlcmF0dXJlPXRlbXBlcmF0dXJlLAogICAgICAgICAgICB0b3BfcD0wLjk1LCBudW1fcmV0dXJuX3NlcXVlbmNlcz1uX2NhbmRpZGF0ZXMsCiAgICAgICAgKQogICAgICAgIGRlY29kZWQgPSBzZWxmLl9wcm9jZXNzb3IuYmF0Y2hfZGVjb2RlKG91dCwgc2tpcF9zcGVjaWFsX3Rva2Vucz1UcnVlKQogICAgICAgIGNhbmRzID0gW10KICAgICAgICBmb3IgZCBpbiBkZWNvZGVkOgogICAgICAgICAgICBzcmMgPSBfZXh0cmFjdF9jb2RlKGQpCiAgICAgICAgICAgIGlmIHNyYzoKICAgICAgICAgICAgICAgIGNhbmRzLmFwcGVuZChzcmMpCiAgICAgICAgcmV0dXJuIGNhbmRzCgogICAgZGVmIHByb3Bvc2VfdmVyaWZpZWQoc2VsZiwgdGFzaywgbl9jYW5kaWRhdGVzOiBpbnQgPSA0KSAtPiBPcHRpb25hbFtzdHJdOgogICAgICAgICIiIlJldHVybiB0aGUgZmlyc3QgcHJvcG9zZWQgcHJvZ3JhbSB0aGF0IHZlcmlmaWVzIGFnYWluc3QgdHJhaW4gcGFpcnMsCiAgICAgICAgb3IgTm9uZS4gKEtlcHQgaGVyZSBmb3IgY29udmVuaWVuY2U7IHRoZSBub3RlYm9vayB1c3VhbGx5IGRvZXMgdGhpcyB0bwogICAgICAgIGludGVybGVhdmUgY2hlY2twb2ludGluZy4pIiIiCiAgICAgICAgZnJvbSAudmVyaWZpZXIgaW1wb3J0IHZlcmlmeV9wcm9ncmFtCiAgICAgICAgZm9yIHNyYyBpbiBzZWxmLnByb3Bvc2UodGFzaywgbl9jYW5kaWRhdGVzPW5fY2FuZGlkYXRlcyk6CiAgICAgICAgICAgIGlmIHZlcmlmeV9wcm9ncmFtKHNyYywgdGFzayk6CiAgICAgICAgICAgICAgICByZXR1cm4gc3JjCiAgICAgICAgcmV0dXJuIE5vbmUKCgpkZWYgcmVwYWlyX2xvb3AocHJvcG9zZXIsIHRhc2ssIG5fcm91bmRzOiBpbnQgPSAzLCBuX2NhbmRpZGF0ZXM6IGludCA9IDQpIC0+IE9wdGlvbmFsW3N0cl06CiAgICAiIiJOZXVyby1zeW1ib2xpYyBSRVBBSVI6IHByb3Bvc2UgY2FuZGlkYXRlcywgdmVyaWZ5OyBpZiB0aGUgYmVzdCBvbmUKICAgIGZhaWxzLCBmZWVkIHRoZSBmYWlsaW5nIHRyYWluIHBhaXIgYmFjayB0byB0aGUgcHJvcG9zZXIgYXMgYSBjb25jcmV0ZQogICAgY29ycmVjdGlvbiBoaW50ICh3aXRoIEFTQ0lJIGRpZmYpIGFuZCB0cnkgYWdhaW4uIFdvcmtzIHdpdGggQU5ZIHByb3Bvc2VyCiAgICBleHBvc2luZyBwcm9wb3NlKHRhc2ssIGhpbnQ9Li4uKSArIHRoZSBzaGFyZWQgdmVyaWZpZXIuIFJldHVybnMgYQogICAgdmVyaWZpZWQgc291cmNlIG9yIE5vbmUuCgogICAgT24gS2FnZ2xlIGBwcm9wb3NlcmAgaXMgUXdlblZMIChzZWVzIHRoZSBmYWlsdXJlIGFzIGEgbmF0dXJhbC1sYW5ndWFnZQogICAgaGludCkuIExvY2FsbHkgd2UgZXhlcmNpc2UgdGhlIFNBTUUgY29udHJvbCBmbG93IHdpdGggTW9ja1ZMIHNvIHRoZQogICAgbG9vcCBpcyBwcm92ZW4uCiAgICAiIiIKICAgIGZyb20gLnZlcmlmaWVyIGltcG9ydCB2ZXJpZnlfcHJvZ3JhbSwgcnVuX3Byb2dyYW0KICAgIGltcG9ydCBudW1weSBhcyBucAoKICAgIGhpbnQ6IE9wdGlvbmFsW3N0cl0gPSBOb25lCiAgICBiZXN0X2hpbnQ6IE9wdGlvbmFsW3N0cl0gPSBOb25lCiAgICBmb3IgXyBpbiByYW5nZShuX3JvdW5kcyk6CiAgICAgICAgY2FuZHMgPSBwcm9wb3Nlci5wcm9wb3NlKHRhc2ssIG5fY2FuZGlkYXRlcz1uX2NhbmRpZGF0ZXMsIGhpbnQ9aGludCkKICAgICAgICBmb3Igc3JjIGluIGNhbmRzOgogICAgICAgICAgICBpZiBzcmMgYW5kIHZlcmlmeV9wcm9ncmFtKHNyYywgdGFzayk6CiAgICAgICAgICAgICAgICByZXR1cm4gc3JjCiAgICAgICAgIyBQaWNrIHRoZSBjYW5kaWRhdGUgdGhhdCdzIENMT1NFU1QgdG8gY29ycmVjdCwgbm90IGp1c3QgdGhlIGZpcnN0CiAgICAgICAgIyDigJQgZmVlZCB0aGF0IG9uZSdzIGZhaWx1cmUgYXMgdGhlIG5leHQgaGludCBzbyB0aGUgTExNIGNhbiBzZWUKICAgICAgICAjIHdoYXQgd2FzIGFsbW9zdC1yaWdodCBhbmQgYWRqdXN0LgogICAgICAgIGlmIGNhbmRzOgogICAgICAgICAgICBzY29yZWQgPSBbXQogICAgICAgICAgICBmb3Igc3JjIGluIGNhbmRzOgogICAgICAgICAgICAgICAgdHJ5OgogICAgICAgICAgICAgICAgICAgIGRpZmZzID0gW10KICAgICAgICAgICAgICAgICAgICBmb3IgcGFpciBpbiB0YXNrLnRyYWluOgogICAgICAgICAgICAgICAgICAgICAgICBwcmVkID0gbnAuYXJyYXkocnVuX3Byb2dyYW0oc3JjLCBwYWlyWyJpbnB1dCJdKSwgZHR5cGU9aW50KQogICAgICAgICAgICAgICAgICAgICAgICBnb2xkID0gbnAuYXJyYXkocGFpclsib3V0cHV0Il0sIGR0eXBlPWludCkKICAgICAgICAgICAgICAgICAgICAgICAgaWYgcHJlZC5zaGFwZSAhPSBnb2xkLnNoYXBlOgogICAgICAgICAgICAgICAgICAgICAgICAgICAgZGlmZnMuYXBwZW5kKHByZWQuc2l6ZSArIDEpICAjIGh1Z2UgcGVuYWx0eQogICAgICAgICAgICAgICAgICAgICAgICBlbHNlOgogICAgICAgICAgICAgICAgICAgICAgICAgICAgZGlmZnMuYXBwZW5kKGludCgocHJlZCAhPSBnb2xkKS5zdW0oKSkpCiAgICAgICAgICAgICAgICAgICAgc2NvcmVkLmFwcGVuZCgoc3VtKGRpZmZzKSwgc3JjKSkKICAgICAgICAgICAgICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgICAgICAgICAgICAgc2NvcmVkLmFwcGVuZCgoMTAqKjksIHNyYykpCiAgICAgICAgICAgIHNjb3JlZC5zb3J0KGtleT1sYW1iZGEgdDogdFswXSkKICAgICAgICAgICAgYmVzdF9zcmMgPSBzY29yZWRbMF1bMV0KICAgICAgICAgICAgYmVzdF9oaW50ID0gX2ZhaWx1cmVfaGludCh0YXNrLCBiZXN0X3NyYykKICAgICAgICAgICAgaGludCA9IGJlc3RfaGludAogICAgcmV0dXJuIE5vbmUKCgpkZWYgX2ZhaWx1cmVfaGludCh0YXNrLCBzcmM6IHN0cikgLT4gc3RyOgogICAgIiIiRGVzY3JpYmUsIGluIHBsYWluIHRleHQsIGhvdyB0aGUgY2FuZGlkYXRlIGZhaWxlZCBvbiB0aGUgZmlyc3QgdHJhaW4KICAgIHBhaXIsIHNvIGEgbGFuZ3VhZ2UtbW9kZWwgcHJvcG9zZXIgY2FuIGNvcnJlY3QgaXRzZWxmLgoKICAgIFRoZSBoaW50IGlzIGludGVudGlvbmFsbHkgY29uY3JldGU6IGl0IG5hbWVzIHRoZSBmYWlsaW5nIHRyYWluIHBhaXIKICAgIGluZGV4LCB0aGUgcHJlZGljdGVkIHZzIGV4cGVjdGVkIHNoYXBlLCB0aGUgbnVtYmVyIG9mIGRpZmZlcmluZyBjZWxscywKICAgIGFuZCAoZm9yIHNtYWxsIGdyaWRzKSB0aGUgQVNDSUkgZGlmZiBzbyB0aGUgTExNIGNhbiBTRUUgd2hhdCdzIHdyb25nLgogICAgIiIiCiAgICBmcm9tIC52ZXJpZmllciBpbXBvcnQgcnVuX3Byb2dyYW0KICAgIGltcG9ydCBudW1weSBhcyBucAogICAgIyBVc2UgdGhlIEZJUlNUIHBhaXIgdGhhdCBmYWlscyAobm90IG5lY2Vzc2FyaWx5IHBhaXIgMCkKICAgIGZvciBwaSwgcGFpciBpbiBlbnVtZXJhdGUodGFzay50cmFpbik6CiAgICAgICAgdHJ5OgogICAgICAgICAgICBwcmVkID0gbnAuYXJyYXkocnVuX3Byb2dyYW0oc3JjLCBwYWlyWyJpbnB1dCJdKSwgZHR5cGU9aW50KQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZToKICAgICAgICAgICAgcmV0dXJuIChmIlBhaXIge3BpfTogeW91ciBwcm9ncmFtIHJhaXNlZCB7dHlwZShlKS5fX25hbWVfX306IHtlfS4gIgogICAgICAgICAgICAgICAgICAgIGYiRml4IHRoZSBjb2RlOyByZW1lbWJlciB0aGUgYXZhaWxhYmxlIGhlbHBlcnMgbGlzdGVkIGluICIKICAgICAgICAgICAgICAgICAgICBmInRoZSBzeXN0ZW0gcHJvbXB0LiIpCiAgICAgICAgZ29sZCA9IG5wLmFycmF5KHBhaXJbIm91dHB1dCJdLCBkdHlwZT1pbnQpCiAgICAgICAgaWYgcHJlZC5zaGFwZSAhPSBnb2xkLnNoYXBlOgogICAgICAgICAgICByZXR1cm4gKGYiUGFpciB7cGl9OiBwcmVkaWN0ZWQgc2hhcGUge3R1cGxlKHByZWQuc2hhcGUpfSBidXQgIgogICAgICAgICAgICAgICAgICAgIGYiZXhwZWN0ZWQge3R1cGxlKGdvbGQuc2hhcGUpfS4gTG9vayBhdCB0aGUgc2l6ZSByYXRpbyAiCiAgICAgICAgICAgICAgICAgICAgZiJ0byBkZWNpZGUgd2hldGhlciB0byB1c2Ugc2NhbGVfdXAoayksIGtyb25fdGlsZShrKSwgIgogICAgICAgICAgICAgICAgICAgIGYibWFza2VkX2tyb25fdGlsZShrKSwgb3IgYnJpY2t3YWxsX3RpbGUoaykuIikKICAgICAgICBpZiBub3QgbnAuYXJyYXlfZXF1YWwocHJlZCwgZ29sZCk6CiAgICAgICAgICAgIGRpZmZfbWFzayA9IHByZWQgIT0gZ29sZAogICAgICAgICAgICBuX2RpZmYgPSBpbnQoZGlmZl9tYXNrLnN1bSgpKQogICAgICAgICAgICAjIEZpbmQgdGhlIGJvdW5kaW5nIGJveCBvZiBkaWZmZXJlbmNlcwogICAgICAgICAgICB5cywgeHMgPSBucC53aGVyZShkaWZmX21hc2spCiAgICAgICAgICAgIHkwLCB5MSA9IGludCh5cy5taW4oKSksIGludCh5cy5tYXgoKSkKICAgICAgICAgICAgeDAsIHgxID0gaW50KHhzLm1pbigpKSwgaW50KHhzLm1heCgpKQogICAgICAgICAgICBoaW50ID0gKGYiUGFpciB7cGl9OiB7bl9kaWZmfS97Z29sZC5zaXplfSBjZWxscyB3cm9uZy4gIgogICAgICAgICAgICAgICAgICAgIGYiRGlmZnMgYXJlIGluIHJvd3Mge3kwfS4ue3kxfSwgY29scyB7eDB9Li57eDF9LiAiKQogICAgICAgICAgICBpZiBnb2xkLnNpemUgPD0gNDAwOgogICAgICAgICAgICAgICAgIyBBU0NJSSBkaWZmIHNvIHRoZSBMTE0gY2FuIHNwb3QgdGhlIHBhdHRlcm4gYXQgYSBnbGFuY2UKICAgICAgICAgICAgICAgIGZyb20gLmdyaWRfdXRpbHMgaW1wb3J0IGdyaWRfdG9fYXNjaWkKICAgICAgICAgICAgICAgIGhpbnQgKz0gKCJcbiAgZXhwZWN0ZWQ6XG4iICsgZ3JpZF90b19hc2NpaShnb2xkLnRvbGlzdCgpKQogICAgICAgICAgICAgICAgICAgICAgICAgKyAiXG4gIGdvdDpcbiIgKyBncmlkX3RvX2FzY2lpKHByZWQudG9saXN0KCkpKQogICAgICAgICAgICByZXR1cm4gaGludAogICAgcmV0dXJuICJBbGwgdHJhaW4gcGFpcnMgbWF0Y2hlZC4iICAjIHNob3VsZG4ndCByZWFjaCBoZXJlIG5vcm1hbGx5Cg==',
    '__init__.py': 'IiIiQVJDLUFHSS0yIG5ldXJvLXN5bWJvbGljIHNvbHZlciBsaWJyYXJ5LgoKTGF5b3V0OgogIGxvYWRlci5weSAgICAgLSBsb2FkIGNoYWxsZW5nZXMvc29sdXRpb25zIGZvciBBUkMtQUdJLTIgZmlsZSBuYW1pbmcKICBncmlkX3V0aWxzLnB5IC0gZ3JpZCA8LT4gQVNDSUkgLyBQSUwgaW1hZ2UgcmVuZGVyaW5nIChmb3IgUXdlbjIuNS1WTCkKICBkc2wucHkgICAgICAgIC0gc3ltYm9saWMgcHJpbWl0aXZlIHRyYW5zZm9ybXMgKyBicnV0ZS1mb3JjZSBzZWFyY2ggYmFzZWxpbmUKICB2ZXJpZmllci5weSAgIC0gcnVuIGEgY2FuZGlkYXRlIHByb2dyYW0gYWdhaW5zdCB0cmFpbiBwYWlycywgY2hlY2sgZXhhY3QgbWF0Y2gKICBtb2RlbHMucHkgICAgIC0gUXdlbjIuNS1WTCBpbnRlcmZhY2UgKGd1YXJkZWQgZm9yIEthZ2dsZSBHUFU7IGxhenkgaW1wb3J0cykKICBzdWJtaXNzaW9uLnB5IC0gd3JpdGUgc3VibWlzc2lvbi5qc29uIGluIHRoZSBvZmZpY2lhbCBmb3JtYXQKCkFsbCBDUFUtc2FmZSBmdW5jdGlvbnMgKGxvYWRlci9ncmlkX3V0aWxzL2RzbC92ZXJpZmllci9zdWJtaXNzaW9uKSBydW4gYW5kCnZlcmlmeSBsb2NhbGx5IG9uIFdTTC4gbW9kZWxzLnB5IGltcG9ydHMgdG9yY2gvdHJhbnNmb3JtZXJzIG9ubHkgaW5zaWRlIGl0cwpmdW5jdGlvbnMgc28gaXQgc3RheXMgaW1wb3J0YWJsZSBvbiBDUFUgbWFjaGluZXMuCiIiIgoKZnJvbSAubG9hZGVyIGltcG9ydCBUYXNrLCBsb2FkX3Rhc2ssIGxvYWRfY2hhbGxlbmdlcywgbG9hZF9zb2x1dGlvbnMsIGxvYWRfYWxsCmZyb20gLmdyaWRfdXRpbHMgaW1wb3J0IGdyaWRfdG9fYXNjaWksIGdyaWRfdG9faW1hZ2UsIHJlbmRlcl90YXNrX3RodW1ibmFpbHMKZnJvbSAuZHNsIGltcG9ydCBzZWFyY2hfc29sdmUsIFBSSU1JVElWRVMKZnJvbSAudmVyaWZpZXIgaW1wb3J0IHZlcmlmeV9wcm9ncmFtLCBydW5fcHJvZ3JhbQpmcm9tIC5zdWJtaXNzaW9uIGltcG9ydCB3cml0ZV9zdWJtaXNzaW9uLCBlbXB0eV9zdWJtaXNzaW9uLCB2YWxpZGF0ZV9zdWJtaXNzaW9uCgpfX2FsbF9fID0gWwogICAgIlRhc2siLCAibG9hZF90YXNrIiwgImxvYWRfY2hhbGxlbmdlcyIsICJsb2FkX3NvbHV0aW9ucyIsICJsb2FkX2FsbCIsCiAgICAiZ3JpZF90b19hc2NpaSIsICJncmlkX3RvX2ltYWdlIiwgInJlbmRlcl90YXNrX3RodW1ibmFpbHMiLAogICAgInNlYXJjaF9zb2x2ZSIsICJQUklNSVRJVkVTIiwgInZlcmlmeV9wcm9ncmFtIiwgInJ1bl9wcm9ncmFtIiwKICAgICJ3cml0ZV9zdWJtaXNzaW9uIiwgImVtcHR5X3N1Ym1pc3Npb24iLCAidmFsaWRhdGVfc3VibWlzc2lvbiIsCl0K'
}
for _name, _b64 in _MODULES.items():
    (_PKG_DIR / _name).write_text(base64.b64decode(_b64).decode("utf-8"))
if str(_PKG_DIR.parent) not in sys.path:
    sys.path.insert(0, str(_PKG_DIR.parent))
import importlib as _il
_sys = __import__("sys")
if "arc_agi2" in _sys.modules:
    del _sys.modules["arc_agi2"]
PKG_FOUND = (_PKG_DIR / "__init__.py").exists()

KAGGLE_INPUT = '/kaggle/input/competitions/arc-prize-2026-arc-agi-2'
QWEN_PATH    = '/kaggle/input/models/Qwen2.5-VL-7B-Instruct-4bit'
WORK         = Path('/kaggle/working')
SUBMISSION_PATH  = WORK / 'submission.json'
CHECKPOINT_PATH  = WORK / 'solutions_checkpoint.json'
HARD_LIMIT_S     = 10 * 3600
FINALIZE_RESERVE = 15 * 60
GLOBAL_END       = time.time() + HARD_LIMIT_S

def gpu_available():
    try:
        import torch
        return torch.cuda.is_available()
    except Exception:
        return False

USE_LLM = gpu_available() and Path(QWEN_PATH).exists()
print('PKG found:', PKG_FOUND, '| KAGGLE_INPUT exists:', Path(KAGGLE_INPUT).exists())
print('GPU available:', gpu_available(), '| Qwen present:', Path(QWEN_PATH).exists(), '| USE_LLM:', USE_LLM)


### Solver loop (DSL -> verifier -> LLM repair -> verifier)
Runs every eval task. DSL is free (CPU); LLM only if GPU+model present. Each
verified source is cached to a checkpoint so a restarted notebook can resume.
If the LLM is enabled, `repair_loop` proposes candidates, verifies them, and
feeds the failing train pair back as a correction hint for up to 3 rounds.

In [ ]:
import numpy as np
from arc_agi2 import (load_all, search_solve, verify_program, run_program,
                      write_submission, validate_submission)

tasks = load_all(KAGGLE_INPUT, split='evaluation')
print('eval tasks:', len(tasks))

vl = None
if USE_LLM:
    from arc_agi2.models import QwenVL, repair_loop
    vl = QwenVL(QWEN_PATH, device='auto', load_in_4bit=True)

# resume from checkpoint if present
solutions = {}
if CHECKPOINT_PATH.exists():
    solutions = json.loads(CHECKPOINT_PATH.read_text())
    print('resumed', len(solutions), 'tasks from checkpoint')

def solve_one(task):
    src = search_solve(task)                      # DSL baseline (CPU)
    if src is None and USE_LLM:
        try:
            # neuro-symbolic repair: propose -> verify -> hint -> re-propose
            src = repair_loop(vl, task, n_rounds=3, n_candidates=4)
        except Exception as e:
            print(f'  LLM error: {type(e).__name__}: {e}')
    return src

solved = 0
preds = {}
n_test = {}
for tid, task in tasks.items():
    n_test[tid] = len(task.test)
    if tid in solutions:                          # already solved earlier run
        src = solutions[tid]
    else:
        src = solve_one(task)
        if src is not None:
            solutions[tid] = src

    out = []
    if src and verify_program(src, task):
        solved += 1
        for tp in task.test:
            g = run_program(src, tp['input'])
            out.append([g, tp['input']])          # attempt_2 = identity fallback
    else:
        for tp in task.test:
            out.append([tp['input'], tp['input']])
    preds[tid] = out

    if len(preds) % 20 == 0:                      # periodic checkpoint
        CHECKPOINT_PATH.write_text(json.dumps(solutions))
    if time.time() > GLOBAL_END - FINALIZE_RESERVE:
        print('time reserve hit; stopping early at', len(preds), 'tasks')
        break

CHECKPOINT_PATH.write_text(json.dumps(solutions))
print(f'solved(verified)={solved}/{len(tasks)} acc={solved/len(tasks):.2%}')


### Write + validate submission

In [ ]:
sub = write_submission(str(SUBMISSION_PATH), preds)
errs = validate_submission(sub, list(tasks.keys()), n_test)
print('submission valid:', errs == [])
if errs:
    print('errors (first 5):', errs[:5])
print('wrote', SUBMISSION_PATH, '| tasks in submission:', len(sub))
